# NSE Daily Stocks + NIFTY 50 — Full Incremental Sync

Filesystem-first, resumable downloader for the Quant project.

**Storage**
- Stocks: `/content/drive/MyDrive/quant/data/parquet`
- NIFTY 50: `/content/drive/MyDrive/quant/data/indices/nifty50`

**Important behavior**
- Existing Parquet files are authoritative and are never redownloaded.
- The manifest is diagnostic only; it never causes an existing/missing file to be skipped.
- Every missing stock date is queued.
- The complete stock queue is checked immediately before downloading; if any queued file exists, execution stops before the first network request.
- A second per-file check happens immediately before each stock request.
- NIFTY uses the current `getHistoricaldatatabletoString` POST payload discovered from the site's JavaScript.
- NIFTY responses are filtered locally because the endpoint can return records outside the requested range.
- Out-of-range NIFTY records are logged before saving, followed by a hard date-range assertion.


In [ ]:
# 1. Install/import
!pip -q install pyarrow requests pandas tqdm

import io
import json
import random
import time
import zipfile
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import requests


In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. Configuration

BASE_DIR = Path("/content/drive/MyDrive/quant")

STOCK_DIR = BASE_DIR / "data" / "parquet"
NIFTY_DIR = BASE_DIR / "data" / "indices" / "nifty50"
MANIFEST_DIR = BASE_DIR / "data" / "manifests"

STOCK_MANIFEST = MANIFEST_DIR / "nse_stock_download_manifest.jsonl"
NIFTY_MANIFEST = MANIFEST_DIR / "nifty50_download_manifest.jsonl"

for p in [STOCK_DIR, NIFTY_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

START_DATE = date(2011, 1, 1)
END_DATE = date.today()

MAX_RETRIES = 4
REQUEST_SLEEP_SECONDS = 0.20

# The endpoint has been observed returning about 188 records even when
# a shorter end date is requested. Keep chunks comfortably below that.
NIFTY_CHUNK_DAYS = 150

DOWNLOAD_STOCKS = True
DOWNLOAD_NIFTY = True

print("Range:", START_DATE, "->", END_DATE)
print("Stocks:", STOCK_DIR)
print("NIFTY :", NIFTY_DIR)


In [ ]:
# 4. Common helpers

STOCK_COLUMNS = ["date", "symbol", "open", "high", "low", "close", "volume"]

def iso(d):
    return pd.Timestamp(d).date().isoformat()

def date_range_days(start, end):
    cur = pd.Timestamp(start).date()
    end = pd.Timestamp(end).date()
    while cur <= end:
        yield cur
        cur += timedelta(days=1)

def parquet_path(directory, day):
    return Path(directory) / f"{iso(day)}.parquet"

def append_jsonl(path, record):
    with Path(path).open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, default=str) + "\n")

def load_latest_manifest(path):
    latest = {}
    path = Path(path)
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
                if row.get("date"):
                    latest[row["date"]] = row
            except Exception:
                continue
    return latest

def atomic_to_parquet(df, path):
    path = Path(path)
    tmp = path.with_suffix(".parquet.tmp")
    if tmp.exists():
        tmp.unlink()
    df.to_parquet(tmp, index=False, engine="pyarrow")
    if path.exists():
        tmp.unlink()
        raise RuntimeError(f"Refusing to overwrite existing file: {path}")
    tmp.replace(path)

def validate_parquet(path, required_columns):
    try:
        df = pd.read_parquet(path, engine="pyarrow")
        missing = set(required_columns) - set(df.columns)
        if missing:
            return False, f"missing columns: {sorted(missing)}"
        if df.empty:
            return False, "empty parquet"
        return True, f"{len(df):,} rows"
    except Exception as e:
        return False, repr(e)


In [ ]:
# 5. NSE session + Bhavcopy URLs

LEGACY_CUTOFF = date(2024, 7, 5)

def build_nse_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/140.0.0.0 Safari/537.36"
        ),
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    })
    try:
        r = s.get("https://www.nseindia.com", timeout=30)
        print("NSE session:", r.status_code)
    except Exception as e:
        print("NSE session warning:", repr(e))
    return s

nse_session = build_nse_session()

def nse_stock_url(day):
    day = pd.Timestamp(day).date()

    if day <= LEGACY_CUTOFF:
        mon = day.strftime("%b").upper()
        filename = f"cm{day.strftime('%d')}{mon}{day.year}bhav.csv.zip"
        return (
            "https://nsearchives.nseindia.com/content/historical/"
            f"EQUITIES/{day.year}/{mon}/{filename}"
        )

    filename = f"BhavCopy_NSE_CM_0_0_0_{day.strftime('%Y%m%d')}_F_0000.csv.zip"
    return f"https://nsearchives.nseindia.com/content/cm/{filename}"

print(nse_stock_url(date(2024, 7, 5)))
print(nse_stock_url(date(2024, 7, 8)))


In [ ]:
# 6. Parse NSE Bhavcopy into the Quant schema

def normalize_stock_bhavcopy(raw_bytes, day):
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as z:
        csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError(f"No CSV inside ZIP: {z.namelist()}")
        data = z.read(csvs[0])

    try:
        text = data.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = data.decode("latin-1")

    df = pd.read_csv(io.StringIO(text))
    df.columns = [
        str(c).strip().upper().replace(" ", "_")
        for c in df.columns
    ]

    def col(*names):
        for name in names:
            if name in df.columns:
                return name
        return None

    symbol = col("SYMBOL")
    series = col("SERIES")
    open_ = col("OPEN")
    high = col("HIGH")
    low = col("LOW")
    close = col("CLOSE")
    volume = col(
        "TOTTRDQTY",
        "TTL_TRD_QNTY",
        "TOTAL_TRADES_QUANTITY",
        "TOTAL_TRADE_QUANTITY",
    )

    missing = [
        name for name, value in {
            "SYMBOL": symbol,
            "OPEN": open_,
            "HIGH": high,
            "LOW": low,
            "CLOSE": close,
            "VOLUME": volume,
        }.items() if value is None
    ]

    if missing:
        raise RuntimeError(
            f"Could not map columns {missing}. "
            f"Actual columns: {list(df.columns)}"
        )

    out = pd.DataFrame({
        "date": pd.Timestamp(day).normalize(),
        "symbol": df[symbol].astype(str).str.strip(),
        "open": pd.to_numeric(df[open_], errors="coerce"),
        "high": pd.to_numeric(df[high], errors="coerce"),
        "low": pd.to_numeric(df[low], errors="coerce"),
        "close": pd.to_numeric(df[close], errors="coerce"),
        "volume": pd.to_numeric(df[volume], errors="coerce"),
    })

    if series:
        eq = df[series].astype(str).str.strip().str.upper().eq("EQ")
        out = out.loc[eq].copy()

    out = out.dropna(subset=["symbol", "open", "high", "low", "close"])

    bad = (
        (out["high"] < out["low"]) |
        (out["high"] < out["open"]) |
        (out["high"] < out["close"]) |
        (out["low"] > out["open"]) |
        (out["low"] > out["close"]) |
        (out[["open", "high", "low", "close"]] <= 0).any(axis=1)
    )

    if bad.any():
        raise RuntimeError(
            f"Invalid OHLC rows for {day}: {int(bad.sum())}"
        )

    return out[STOCK_COLUMNS].sort_values("symbol").reset_index(drop=True)


In [ ]:
# 7. Download one stock date

def download_stock_day(day):
    day = pd.Timestamp(day).date()
    path = parquet_path(STOCK_DIR, day)

    if path.exists():
        raise RuntimeError(
            f"ABORT: stock file appeared before network request: {path}"
        )

    url = nse_stock_url(day)
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = nse_session.get(url, timeout=60)

            if r.status_code == 404:
                return {
                    "status": "not_found",
                    "date": iso(day),
                    "http_status": 404,
                    "url": url,
                }

            r.raise_for_status()

            if len(r.content) < 100:
                raise RuntimeError(
                    f"Suspiciously small response: {len(r.content)} bytes"
                )

            df = normalize_stock_bhavcopy(r.content, day)

            if path.exists():
                raise RuntimeError(
                    f"ABORT: stock file appeared before save: {path}"
                )

            atomic_to_parquet(df, path)

            return {
                "status": "downloaded",
                "date": iso(day),
                "rows": len(df),
                "bytes": len(r.content),
                "http_status": r.status_code,
                "url": url,
            }

        except Exception as e:
            last_error = repr(e)
            if attempt < MAX_RETRIES:
                time.sleep(min(30, 2 ** (attempt - 1) + random.random()))

    return {
        "status": "failed",
        "date": iso(day),
        "message": last_error,
        "url": url,
    }


In [ ]:
# 8. Filesystem-first stock inventory

all_dates = list(date_range_days(START_DATE, END_DATE))

stock_existing = [
    d for d in all_dates
    if parquet_path(STOCK_DIR, d).exists()
]

stock_missing = [
    d for d in all_dates
    if not parquet_path(STOCK_DIR, d).exists()
]

# Every missing date must have no Parquet.
assert all(
    not parquet_path(STOCK_DIR, d).exists()
    for d in stock_missing
)

assert len(stock_existing) + len(stock_missing) == len(all_dates)

print("=" * 90)
print("STOCK FILESYSTEM INVENTORY")
print("=" * 90)
print("Calendar dates checked:", len(all_dates))
print("Existing Parquets     :", len(stock_existing))
print("Missing Parquets      :", len(stock_missing))


In [ ]:
# 9. Manifest classification — diagnostics only

manifest = load_latest_manifest(STOCK_MANIFEST)

manifest_downloaded_but_missing = [
    d for d in stock_missing
    if manifest.get(iso(d), {}).get("status") == "downloaded"
]

manifest_not_found = [
    d for d in stock_missing
    if manifest.get(iso(d), {}).get("status") == "not_found"
]

manifest_failed = [
    d for d in stock_missing
    if manifest.get(iso(d), {}).get("status") == "failed"
]

# IMPORTANT: filesystem is authoritative.
download_queue = list(stock_missing)

assert len(download_queue) == len(stock_missing)

assert all(
    not parquet_path(STOCK_DIR, d).exists()
    for d in download_queue
)

print("=" * 90)
print("STOCK DOWNLOAD QUEUE")
print("=" * 90)
print("Queue size:", len(download_queue))
print("Manifest says downloaded but missing:", len(manifest_downloaded_but_missing))
print("Manifest says not_found:", len(manifest_not_found))
print("Manifest says failed:", len(manifest_failed))


In [ ]:
# 10. FINAL STOCK PREFLIGHT — must pass before ANY network download

if DOWNLOAD_STOCKS and download_queue:
    existing_in_queue = [
        str(parquet_path(STOCK_DIR, d))
        for d in download_queue
        if parquet_path(STOCK_DIR, d).exists()
    ]

    if existing_in_queue:
        raise RuntimeError(
            "ABORTED BEFORE DOWNLOAD: queue contains existing files:\n"
            + "\n".join(existing_in_queue[:100])
        )

    assert all(
        not parquet_path(STOCK_DIR, d).exists()
        for d in download_queue
    )

    print(f"✓ Stock preflight passed: {len(download_queue):,} files absent.")
else:
    print("No stock downloads required.")


In [ ]:
# 11. Download stock queue

stock_results = []

if DOWNLOAD_STOCKS:
    for i, day in enumerate(download_queue, 1):

        # Immediate check before every request.
        path = parquet_path(STOCK_DIR, day)
        if path.exists():
            raise RuntimeError(
                f"ABORT: queued stock file appeared before request: {path}"
            )

        result = download_stock_day(day)
        stock_results.append(result)
        append_jsonl(STOCK_MANIFEST, result)

        print(
            f"[{i:,}/{len(download_queue):,}] "
            f"{day} -> {result['status']}"
        )

        time.sleep(REQUEST_SLEEP_SECONDS)

print("Stock download complete.")


In [ ]:
# 12. NIFTY session

NIFTY_ENDPOINT = (
    "https://www.niftyindices.com/BackPage/"
    "getHistoricaldatatabletoString"
)

NIFTY_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/json; charset=utf-8",
    "Origin": "https://www.niftyindices.com",
    "Referer": "https://www.niftyindices.com/reports",
}

nifty_session = requests.Session()
nifty_session.headers.update(NIFTY_HEADERS)

r = nifty_session.get("https://www.niftyindices.com/reports", timeout=30)
print("NIFTY reports page:", r.status_code)


In [ ]:
# 13. NIFTY fetch with out-of-range diagnostics and hard filtering

def fetch_nifty_range(session, requested_start, requested_end, sample_rows=20):
    requested_start = pd.Timestamp(requested_start).normalize()
    requested_end = pd.Timestamp(requested_end).normalize()

    start_str = requested_start.strftime("%d-%m-%Y")
    end_str = requested_end.strftime("%d-%m-%Y")

    cinfo = (
        "{'name':'NIFTY 50',"
        f"'startDate':'{start_str}',"
        f"'endDate':'{end_str}',"
        "'indexName':'NIFTY 50'}"
    )

    payload = {"cinfo": cinfo}

    print("=" * 90)
    print("NIFTY ENDPOINT DIAGNOSTIC")
    print("=" * 90)
    print(f"Requested range : {requested_start.date()} -> {requested_end.date()}")
    print(f"Endpoint        : {NIFTY_ENDPOINT}")

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.post(
                NIFTY_ENDPOINT,
                headers=NIFTY_HEADERS,
                json=payload,
                timeout=60,
            )

            print("HTTP status     :", response.status_code)
            print("Content-Type    :", response.headers.get("Content-Type"))
            print("Response size   :", f"{len(response.content):,}", "bytes")

            response.raise_for_status()

            try:
                records = response.json()
            except Exception as e:
                raise RuntimeError(
                    "NIFTY endpoint did not return JSON. "
                    f"First 500 chars: {response.text[:500]!r}"
                ) from e

            if not isinstance(records, list):
                raise RuntimeError(
                    f"Unexpected response type: {type(records)}"
                )

            print("Records returned:", f"{len(records):,}")

            if not records:
                return pd.DataFrame(columns=STOCK_COLUMNS)

            df = pd.DataFrame(records)

            required = {
                "HistoricalDate", "OPEN", "HIGH", "LOW", "CLOSE"
            }
            missing = required - set(df.columns)
            if missing:
                raise RuntimeError(
                    f"Missing NIFTY columns: {sorted(missing)}"
                )

            df["date"] = pd.to_datetime(
                df["HistoricalDate"],
                format="%d %b %Y",
                errors="coerce",
            ).dt.normalize()

            valid = df["date"].notna()

            if valid.any():
                actual_min = df.loc[valid, "date"].min()
                actual_max = df.loc[valid, "date"].max()

                print("\nReturned date range:")
                print("  Earliest returned:", actual_min.date())
                print("  Latest returned  :", actual_max.date())

                before = df[valid & (df["date"] < requested_start)].copy()
                after = df[valid & (df["date"] > requested_end)].copy()
                out = pd.concat([before, after], ignore_index=True)

                print("\n" + "-" * 90)
                print("OUT-OF-RANGE RECORD DIAGNOSTIC")
                print("-" * 90)

                if out.empty:
                    print("✓ No out-of-range records returned.")
                else:
                    print(
                        f"WARNING: {len(out):,} out-of-range records returned."
                    )
                    print("  Before requested start:", len(before))
                    print("  After requested end   :", len(after))

                    cols = [
                        "HistoricalDate", "date",
                        "OPEN", "HIGH", "LOW", "CLOSE"
                    ]
                    cols = [c for c in cols if c in out.columns]

                    print("\nSample out-of-range records:")
                    print(
                        out.sort_values("date")[cols]
                        .head(sample_rows)
                        .to_string(index=False)
                    )

                    if not before.empty:
                        print(
                            "Earliest out-of-range:",
                            before["date"].min().date()
                        )
                    if not after.empty:
                        print(
                            "Latest out-of-range:",
                            after["date"].max().date()
                        )

            in_range = (
                valid
                & (df["date"] >= requested_start)
                & (df["date"] <= requested_end)
            )

            filtered = df.loc[in_range].copy()

            print("\n" + "-" * 90)
            print("FILTER RESULT")
            print("-" * 90)
            print("Returned records :", len(df))
            print("In-range records :", len(filtered))
            print("Removed records  :", len(df) - len(filtered))

            if filtered.empty:
                return pd.DataFrame(columns=STOCK_COLUMNS)

            filtered["symbol"] = "NIFTY50"
            filtered["open"] = pd.to_numeric(filtered["OPEN"], errors="coerce")
            filtered["high"] = pd.to_numeric(filtered["HIGH"], errors="coerce")
            filtered["low"] = pd.to_numeric(filtered["LOW"], errors="coerce")
            filtered["close"] = pd.to_numeric(filtered["CLOSE"], errors="coerce")
            filtered["volume"] = pd.NA

            filtered = filtered[
                STOCK_COLUMNS
            ].sort_values("date").reset_index(drop=True)

            # HARD RANGE ASSERTION BEFORE ANY SAVE.
            bad = filtered[
                (filtered["date"] < requested_start) |
                (filtered["date"] > requested_end)
            ]

            if not bad.empty:
                raise RuntimeError(
                    "CRITICAL: out-of-range records survived filtering."
                )

            assert filtered["date"].min() >= requested_start
            assert filtered["date"].max() <= requested_end

            print(
                f"✓ Date-range validation passed: {len(filtered):,} "
                f"records strictly within "
                f"{requested_start.date()} -> {requested_end.date()}"
            )

            return filtered

        except Exception as e:
            last_error = repr(e)
            if attempt < MAX_RETRIES:
                wait = min(30, 2 ** (attempt - 1) + random.random())
                print(f"Attempt {attempt} failed; retrying in {wait:.1f}s")
                time.sleep(wait)

    raise RuntimeError(
        f"NIFTY request failed after {MAX_RETRIES} attempts: {last_error}"
    )


In [ ]:
# 14. NIFTY filesystem inventory

nifty_all_dates = list(date_range_days(START_DATE, END_DATE))

nifty_existing = [
    d for d in nifty_all_dates
    if parquet_path(NIFTY_DIR, d).exists()
]

nifty_missing = [
    d for d in nifty_all_dates
    if not parquet_path(NIFTY_DIR, d).exists()
]

assert all(
    not parquet_path(NIFTY_DIR, d).exists()
    for d in nifty_missing
)

assert len(nifty_existing) + len(nifty_missing) == len(nifty_all_dates)

print("=" * 90)
print("NIFTY FILESYSTEM INVENTORY")
print("=" * 90)
print("Calendar dates checked:", len(nifty_all_dates))
print("Existing Parquets     :", len(nifty_existing))
print("Missing Parquets      :", len(nifty_missing))


In [ ]:
# 15. NIFTY chunk planner

def make_chunks(start, end, chunk_days=150):
    start = pd.Timestamp(start).date()
    end = pd.Timestamp(end).date()

    chunks = []
    cur = start

    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days - 1), end)
        chunks.append((cur, chunk_end))
        cur = chunk_end + timedelta(days=1)

    return chunks

nifty_chunks = make_chunks(
    START_DATE,
    END_DATE,
    NIFTY_CHUNK_DAYS,
)

print("NIFTY API chunks:", len(nifty_chunks))
print("First:", nifty_chunks[:3])
print("Last :", nifty_chunks[-2:])


In [ ]:
# 16. Download NIFTY missing dates

nifty_results = []

if DOWNLOAD_NIFTY and nifty_missing:

    for chunk_no, (chunk_start, chunk_end) in enumerate(nifty_chunks, 1):

        chunk_missing = [
            d for d in nifty_missing
            if chunk_start <= d <= chunk_end
        ]

        if not chunk_missing:
            continue

        print("\n" + "=" * 90)
        print(
            f"NIFTY CHUNK {chunk_no}/{len(nifty_chunks)}: "
            f"{chunk_start} -> {chunk_end}"
        )
        print("Missing files in chunk:", len(chunk_missing))

        # Pre-request race check.
        races = [
            str(parquet_path(NIFTY_DIR, d))
            for d in chunk_missing
            if parquet_path(NIFTY_DIR, d).exists()
        ]
        if races:
            raise RuntimeError(
                "ABORT: NIFTY queue contains files that appeared after "
                "inventory:\n" + "\n".join(races[:50])
            )

        df = fetch_nifty_range(
            nifty_session,
            chunk_start,
            chunk_end,
        )

        if df.empty:
            print("No in-range NIFTY records returned.")
            continue

        saved = 0

        for day, day_df in df.groupby("date", sort=True):
            day = pd.Timestamp(day).date()
            path = parquet_path(NIFTY_DIR, day)

            # Only save a date that was missing when this run started.
            if day not in chunk_missing:
                continue

            if path.exists():
                raise RuntimeError(
                    f"ABORT: NIFTY file appeared before save: {path}"
                )

            if len(day_df) != 1:
                raise RuntimeError(
                    f"Expected 1 NIFTY50 row for {day}, got {len(day_df)}"
                )

            day_df = day_df[STOCK_COLUMNS].copy()
            atomic_to_parquet(day_df, path)

            result = {
                "status": "downloaded",
                "date": iso(day),
                "rows": len(day_df),
                "source_start": iso(chunk_start),
                "source_end": iso(chunk_end),
            }

            nifty_results.append(result)
            append_jsonl(NIFTY_MANIFEST, result)
            saved += 1

        print("Saved:", saved)
        time.sleep(REQUEST_SLEEP_SECONDS)

else:
    print("No NIFTY downloads required.")


In [ ]:
# 17. Final validation and summary

stock_downloaded = [
    r for r in stock_results
    if r["status"] == "downloaded"
]

stock_not_found = [
    r for r in stock_results
    if r["status"] == "not_found"
]

stock_failed = [
    r for r in stock_results
    if r["status"] == "failed"
]

nifty_downloaded = [
    r for r in nifty_results
    if r["status"] == "downloaded"
]

print("=" * 90)
print("DOWNLOAD SUMMARY")
print("=" * 90)
print("\nSTOCKS")
print("Downloaded:", len(stock_downloaded))
print("Not found :", len(stock_not_found))
print("Failed    :", len(stock_failed))

if stock_failed:
    print("\nFailed stock dates:")
    for r in stock_failed[:100]:
        print(r["date"], r.get("message"))

print("\nNIFTY 50")
print("Downloaded:", len(nifty_downloaded))

stock_files = sum(
    parquet_path(STOCK_DIR, d).exists()
    for d in all_dates
)

nifty_files = sum(
    parquet_path(NIFTY_DIR, d).exists()
    for d in nifty_all_dates
)

print("\nFINAL FILE COUNTS")
print("Stock Parquets:", stock_files)
print("Stock missing :", len(all_dates) - stock_files)
print("NIFTY Parquets:", nifty_files)
print("NIFTY missing :", len(nifty_all_dates) - nifty_files)


In [ ]:
# 18. Validate all existing Parquets

stock_bad = []
nifty_bad = []

for d in all_dates:
    p = parquet_path(STOCK_DIR, d)
    if p.exists():
        ok, msg = validate_parquet(p, STOCK_COLUMNS)
        if not ok:
            stock_bad.append((iso(d), msg))

for d in nifty_all_dates:
    p = parquet_path(NIFTY_DIR, d)
    if p.exists():
        ok, msg = validate_parquet(p, STOCK_COLUMNS)
        if not ok:
            nifty_bad.append((iso(d), msg))

print("Bad stock files:", len(stock_bad))
print("Bad NIFTY files:", len(nifty_bad))

if stock_bad:
    print("\nBAD STOCK FILES:")
    for x in stock_bad[:50]:
        print(x)

if nifty_bad:
    print("\nBAD NIFTY FILES:")
    for x in nifty_bad[:50]:
        print(x)


## Rerun behavior

The notebook is intentionally **filesystem-first**.

| Filesystem | Manifest | Action |
|---|---|---|
| Exists | downloaded | Skip |
| Exists | failed | Skip |
| Exists | not_found | Skip |
| Exists | absent | Skip |
| Missing | downloaded | Retry |
| Missing | failed | Retry |
| Missing | not_found | Retry |
| Missing | absent | Download |

This means a stale manifest can never hide a missing Parquet file.

For NIFTY, the endpoint can return a wider date range than requested. Those records are logged and filtered locally **before any Parquet is written**.
